# Modern Inference Engines

> The preceding chapters optimized one request: token selection, Decode cost, quantization, and speculation. A production service instead receives many requests at once—short and long outputs, shared system prompts, and 100K-token documents. The engine must decide who runs next and where every cache lives.
>
> This is the work of vLLM, SGLang, and TensorRT-LLM. Continuous Batching, PagedAttention, Prefix Caching, Chunked Prefill, and Prefill–Decode disaggregation each solve a concrete scheduling or memory problem.
>
> We cover metrics, scheduling, paged KV memory, prefix reuse, fairness for long prompts, workload separation, kernels, and multi-GPU terminology.


## 1. Throughput, TTFT, and TPOT

A service needs at least three metrics. **Throughput** is total tokens produced per second across requests and strongly affects cost. **TTFT** is the wait until the first token appears. **TPOT** is the interval between later tokens and determines streaming smoothness.

The metrics trade off. Higher concurrency reuses weight movement and increases throughput, but may worsen per-request TTFT and TPOT. A claim that one engine is “30% faster” is incomplete without the metric and percentile.

```text
Prefill -> mainly TTFT
Decode  -> mainly TPOT
both    -> throughput
```

Track P50, P95, and P99: an average can hide a small set of requests starved for many seconds.


## 2. From Static to Continuous Batching

Decode loads model weights while serving only one new token per request. Combining 32 requests lets one weight read serve 32 tokens. **Batching makes each data movement useful to more requests.** The following calculation compares compute and bandwidth ceilings.


In [ ]:
params = 7e9
gpu_tflops = 312
gpu_bw_gbs = 2000

flops_per_token = 2 * params
compute_tokens_s = gpu_tflops*1e12/flops_per_token

weight_bytes = params*2
bandwidth_single_stream = gpu_bw_gbs*1e9/weight_bytes

print("compute roofline proxy:", round(compute_tokens_s), "token/s")
print("single-stream weight-bandwidth proxy:", round(bandwidth_single_stream), "token/s")
print("Note: this is only an order-of-magnitude intuition for why Decode is often memory-bound.")


The theoretical compute ceiling may exceed 22,000 tokens/s, while a single request limited by moving weights may reach only about 140. Larger batches move toward the former limit.

**Static Batching** starts and finishes a fixed batch together. Short requests wait idly for the longest, and new requests cannot enter. **Continuous Batching** schedules at each iteration: completed requests leave after any Decode step and waiting requests immediately fill their slots.

The simulator uses six requests with generation lengths from 5 to 200 and at most four active slots.


In [ ]:
names = ["A", "B", "C", "D", "E", "F"]
lengths = [5, 200, 8, 150, 6, 180]   # number of tokens each request must generate
batch_size = 4

def simulate_static(lengths, batch_size):
    """Static batching: form a batch and run it together until the longest request finishes."""
    schedule = []
    t = 0
    for i in range(0, len(lengths), batch_size):
        group = lengths[i:i + batch_size]
        for j in range(len(group)):
            schedule.append((i + j, t, t + max(group)))
        t += max(group)
    return schedule

def simulate_continuous(lengths, batch_size):
    """Continuous batching: whenever a slot opens, immediately admit a waiting request."""
    schedule = [None] * len(lengths)
    pending = list(range(len(lengths)))
    remaining = {i: l for i, l in enumerate(lengths)}
    active = {}
    t = 0
    while pending or active:
        while pending and len(active) < batch_size:
            rid = pending.pop(0)
            active[rid] = t
        for rid in list(active):
            remaining[rid] -= 1
            if remaining[rid] == 0:
                schedule[rid] = (rid, active[rid], t + 1)
                del active[rid]
        t += 1
    return schedule

static_sched = simulate_static(lengths, batch_size)
cont_sched = simulate_continuous(lengths, batch_size)

print("Request generation lengths:", dict(zip(names, lengths)), " batch_size =", batch_size)
print()
for name, (rid, s, e) in zip(names, static_sched):
    print(f"Static      {name}: step {s:>3} - {e:>3}")
print("Static total steps:", max(e for _, s, e in static_sched))
print()
for name, (rid, s, e) in zip(names, cont_sched):
    print(f"Continuous  {name}: step {s:>3} - {e:>3}")
print("Continuous total steps:", max(e for _, s, e in cont_sched))
print()
print("Key observation: static batching ties short A and C to 200-token B, so five-token A still waits 200 steps;")
print("In continuous batching, E fills A's slot immediately and F need not wait for a second batch, reducing 380 steps to 200.")


In [ ]:
# Gantt chart: static batch above (aligned completion), continuous batch below (refill upon completion).
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(7, 4.5), sharex=True)
panels = [(axes[0], static_sched, "Static batching (batch locked until longest finishes)"),
          (axes[1], cont_sched, "Continuous batching (slot refilled immediately)")]
for ax, sched, title in panels:
    for rid, s, e in sched:
        ax.barh(rid, e - s, left=s, height=0.6, color="tab:blue", alpha=0.8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10)
axes[1].set_xlabel("decode step")
plt.tight_layout()
plt.show()


In the static Gantt chart, request A needs five steps but occupies a slot until B finishes at 200; E cannot start at step 5. In the continuous chart, E replaces A immediately and F enters without waiting for a second fixed batch. Total time falls from 380 to 200 steps.

The batch becomes a dynamic set of slots rather than a fixed container. Scheduling decides who computes; PagedAttention addresses where each request's KV Cache resides.


## 3. PagedAttention and KV Cache Paging

Each active request has a different cache length. Reserving one contiguous region at maximum context wastes unused capacity and creates fragmentation as differently sized requests enter and leave.

PagedAttention applies the operating-system solution: split KV Cache into fixed-size blocks, allocate on demand, and map logical sequence positions to non-contiguous physical pages. A sequence remains logically continuous without requiring contiguous GPU memory.

The next calculation compares reservation with paging.


In [ ]:
# Simulate two KV Cache allocations: reserve max_len at once versus allocate 16-token pages on demand.
seqs = [("req1", 50), ("req2", 300), ("req3", 120), ("req4", 45)]
max_len = 512            # reserve for the maximum context length in the contiguous scheme
page_size = 16           # PagedAttention-style page size
kb_per_token = 2         # illustrative KV Cache bytes per token

actual = sum(l * kb_per_token for _, l in seqs)
reserved = len(seqs) * max_len * kb_per_token
paged = sum(-(-l // page_size) * page_size * kb_per_token for _, l in seqs)

print(f"KV actually needed:      {actual:>5} KB")
print(f"Contiguous (max_len): {reserved:>5} KB, waste {reserved - actual} KB")
print(f"Paged (16/page):      {paged:>5} KB, waste {paged - actual} KB")
print()
print("Key observation: paging limits waste to less than one page per request;")
print("Smaller pages waste less memory but increase page-table and mapping overhead—the familiar virtual-memory tradeoff.")


In [ ]:
# Compare VRAM actually needed with contiguous reservation and paging.
import matplotlib.pyplot as plt

labels = ["reserved (max_len)", "actual KV needed", "paged (16 tokens/page)"]
values = [reserved / 1024, actual / 1024, paged / 1024]

plt.figure(figsize=(6, 3.2))
plt.bar(labels, values, color=["tab:red", "tab:green", "tab:blue"])
plt.ylabel("KV cache memory (MB)")
plt.title("Paged allocation cuts the waste of reserving max_len")
for i, v in enumerate(values):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.show()


For the same four requests, contiguous reservation wastes more than 3 GB, while paging limits waste to less than one page per request. The recovered memory supports more concurrent slots.

**PagedAttention does not change the Attention formula.** It manages KV Cache memory. FlashAttention instead optimizes reads and writes inside Attention computation; similar names refer to different layers.


## 4. Prefix Caching and RadixAttention

Production requests frequently share a system prompt, document, or conversation history. Recomputing identical Prefill produces identical KV state.

**Prefix Cache** retains that state so later requests skip matching prompt regions. Longer hits reduce TTFT more.

Shared prefixes form a tree: `ABCDEF`, `ABCDXY`, and `ABCZZ` share `ABC`, and the first two share `ABCD`. SGLang's **RadixAttention** stores common token paths once in a radix tree and finds the longest reusable prefix for a new request.


In [ ]:
# Prefix sharing: place three request token sequences in a trie so the common prefix needs Prefill only once.
seqs = [list("ABCDEF"), list("ABCDXY"), list("ABCZZ")]

def count_trie_tokens(seqs):
    'Number of tokens: '
    root, total = {}, 0
    for seq in seqs:
        node = root
        for tok in seq:
            if tok not in node:
                node[tok] = {}
                total += 1      # only the first occurrence of a token needs Prefill computation
            node = node[tok]
    return total

naive_total = sum(len(s) for s in seqs)
radix_total = count_trie_tokens(seqs)

print("Three requests:", ["".join(s) for s in seqs])
print(f"Without sharing: {naive_total} tokens each run their own Prefill")
print(f"After radix sharing: {radix_total} tokens of Prefill (saving {naive_total - radix_total})")
print()
print("Key observation: longer system prompts and more concurrent requests increase the Prefill saved by prefix reuse.")


In [ ]:
# Compare independent Prefill for every request with Prefill work after prefix-tree sharing.
import matplotlib.pyplot as plt

plt.figure(figsize=(4.8, 3))
plt.bar(["no sharing", "radix tree"], [naive_total, radix_total],
        color=["tab:red", "tab:green"])
plt.ylabel("prefill tokens to compute")
plt.title("Shared prefixes are computed once")
for i, v in enumerate([naive_total, radix_total]):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()


A radix tree reduces 18 token positions of separate Prefill work to 10; the saved eight come from shared prefixes. With long system prompts or multi-turn history, each turn may need Prefill only for newly added tokens.


## 5. Chunked Prefill

Continuous Batching can still suffer head-of-line blocking from a 100K-token prompt. Its long Prefill may occupy the GPU for hundreds of milliseconds while active Decode streams pause, creating TPOT tail latency.

**Chunked Prefill** divides the prompt into blocks and interleaves them with Decode steps. The long request performs similar total work, but other requests continue streaming. Smaller chunks protect TPOT but slightly increase the long request's TTFT; larger chunks do the opposite. Engines expose this fairness trade-off through chunk-size settings.


## 6. Prefill / Decode Disaggregation

LLM inference has two phases with **completely opposite resource usage patterns**:

| Phase | Compute characteristic | Bottleneck | Typical operation |
|:---|:---|:---|:---|
| **Prefill** | Process the entire prompt in one shot | Compute-bound (compute) | Process a 1K-32K token input |
| **Decode** | Generate one new token at a time | Memory-bound (memory bandwidth) | Generate output tokens one by one |

The prefill phase saturates compute and underutilizes bandwidth; the decode phase saturates bandwidth and underutilizes compute. If both kinds of requests run in the same cluster, resource conflicts arise — prefill requests and decode requests each wait for resources they do not lack, and GPU utilization stays low.

**Prefill/Decode disaggregation** deploys the two phases in different clusters. The prefill cluster uses high-compute machines (such as H100 SXM5), and the decode cluster uses high-bandwidth machines. After prefill completes, the KV cache is transferred over the network to the decode cluster to continue generation. Mooncake (the inference framework behind Moonshot/Kimi) and DistServe both adopt this architecture, with papers reporting throughput improvements of 50%-150%.

The engineering difficulty is KV cache transfer — a 32K-token KV cache can be several GB, and cross-machine transfer latency cannot be ignored. Mooncake solves this with a global KV cache pool plus RDMA networking.

## 7. FlashAttention, FlashInfer, and CUDA Graphs

The previous mechanisms are primarily system-level scheduling and memory management. Execution-level tools solve other costs:

- **FlashAttention / FlashInfer / FlashMLA** reduce Attention-kernel HBM traffic by tiling and accumulating without materializing large score matrices.
- **Fused kernels** combine small operations, avoiding launch overhead and intermediate writes.
- **CUDA Graphs** capture the nearly repeated Decode kernel sequence and replay it as one graph, reducing per-kernel launch overhead.
- **`torch.compile` and graph optimization** fuse and specialize computation at graph level.

```text
PagedAttention -> KV Cache memory management (system layer)
FlashAttention -> Attention data movement (kernel layer)
CUDA Graph     -> repeated launch overhead (runtime layer)
```


## 8. TP, PP, DP, EP, and CP

| Name | Split dimension | Intuition |
|:---|:---|:---|
| Tensor Parallel (TP) | Matrices within a layer | Several GPUs compute one large layer |
| Pipeline Parallel (PP) | Transformer layers | Different layers live on different GPUs |
| Data Parallel (DP) | Model replicas | Replicas serve more requests |
| Expert Parallel (EP) | MoE experts | Experts live on different GPUs |
| Context Parallel (CP) | Sequence dimension | A very long context spans GPUs |

TP is common in inference (`--tensor-parallel-size 2`). For any scheme, ask where parameters, KV Cache, tokens, and communication live, and whether communication consumes the parallel speedup.


## 9. Terminology Map

| Term | Problem solved |
|:---|:---|
| Continuous Batching | Scheduling: refill slots as short requests finish |
| PagedAttention | Memory: page KV Cache and reduce reservation/fragmentation |
| Prefix Caching | Reuse: avoid repeated Prefill for identical prefixes |
| RadixAttention | Reuse: organize shared prefixes as a tree |
| Chunked Prefill | Fairness: prevent long prompts from blocking Decode |
| PD disaggregation / KV transfer | Resources: separate workloads and transfer KV |
| FlashAttention / FlashInfer | Kernels: improve Attention memory traffic |
| CUDA Graph | Runtime: reduce Decode launch overhead |
| TP / PP / DP / EP / CP | Multi-GPU partition dimensions |
| Speculative Decoding | Serial work: validate several tokens per target forward |
| KV Cache quantization | Memory: store cache at lower precision |

A job description listing these mechanisms should now map to concrete system layers, problems, and metrics.


## Summary

- [ ] Core metrics of LLM serving: throughput, TTFT, TPOT, and concurrency, which constrain each other
- [ ] The bottleneck of LLM inference is memory bandwidth rather than compute — this is why KV cache size matters so much
- [ ] Contiguous allocation under multiple requests causes internal and external fragmentation; PagedAttention solves it with a block table
- [ ] Continuous batching dynamically forms batches at the iteration level, letting short requests finish first and new requests join immediately
- [ ] Prefix caching (RadixAttention) uses a trie to organize request sequences and automatically reuses the KV cache of shared prefixes
- [ ] Prefill is compute-bound and decode is memory-bound; disaggregated deployment (Mooncake/DistServe) significantly improves throughput
- [ ] Mainstream engines: vLLM (general), SGLang (agentic), TensorRT-LLM (peak NVIDIA), LMDeploy (Chinese models)

References: [vLLM/PagedAttention](https://arxiv.org/abs/2309.06180), [SGLang/RadixAttention](https://arxiv.org/abs/2312.07104), [Orca/Continuous batching](https://www.usenix.org/con/osdi22/presentation/yu), [DistServe](https://arxiv.org/abs/2401.09670), [Mooncake](https://arxiv.org/abs/2407.00079).

## Exercises

> You can ask AI to help explain the approach, but it is not recommended to have AI "solve the exercise for you" directly.

**Exercise 1: KV Cache Capacity Calculation**

Given: A100 80GB, Llama-7B FP16 (weights 13 GB), block_size = 16 tokens. The remaining memory is all used for KV cache. Calculate: at 32K context, what is the maximum number of concurrent requests that can be served simultaneously? (Only consider the KV cache limit, ignore other overhead.)

Hint: Refer to the calculation in Section 1. Each concurrent request needs `seq_len x num_layers x num_heads x head_dim x 2 x bytes_per_element` bytes of KV cache. Remaining memory = 80 - 13 = 67 GB.

### Exercise 1: Implement One Continuous-Batching Step

`active` contains running requests as `[id, remaining_tokens]`; `pending` is the queue. Fill an empty slot, decrement every active request, and remove requests reaching zero after the step.

Hint: move `pending[0]` into `active` when capacity exists, then decrement all active entries.


In [ ]:
# Exercise 1: one continuous-batching scheduling step

def one_step(active, pending, batch_size):
    """Return (new active requests, list of request IDs completed in this step)."""
    if pending and len(active) < batch_size:
        # TODO: Replace the triple-quoted content below with your code
        """Remove the first pending request and add it to active."""
    finished = []
    for item in active:
        item[1] -= 1
        if item[1] == 0:
            finished.append(item[0])
    return [item for item in active if item[1] > 0], finished

active, done = one_step([], [[1, 5]], 4)
assert [i[0] for i in active] == [1] and [i[1] for i in active] == [4]
assert done == []
active, done = one_step([[1, 1]], [[2, 3]], 2)
assert done == [1]
assert [i[0] for i in active] == [2] and [i[1] for i in active] == [2]
print("Exercise 1 passed: you implemented the core step of continuous batching")


### Exercise 2: Calculate Paging Waste

A sequence of length `length` with `page_size` occupies `ceil(length/page_size)*page_size` token slots. One extra token may require a whole new page.

Hint: integer ceiling division is `-(-a // b)`.


In [ ]:
# Exercise 2: paging waste

def paged_tokens(length, page_size=16):
    """Return the occupied token slots after paging with page_size."""
    # TODO: Replace the triple-quoted content below with your code
    """Round length / page_size upward, then multiply by page_size."""

assert paged_tokens(16, 16) == 16      # exactly one page, with no waste
assert paged_tokens(17, 16) == 32      # one extra token requires another whole page
assert paged_tokens(45, 16) == 48
waste = sum(paged_tokens(l) - l for l in [50, 300, 120, 45])
assert waste == 29, waste
print("Exercise 2 passed: paging limits waste to less than one page per request")


### Exercise 3: Find the Longest Reusable Prefix in a Radix Tree

Walk the prefix tree for a new request and Prefill only the unmatched suffix.

Hint: start at the root and follow tokens until the current node has no matching child.


In [ ]:
# Exercise 3: longest-prefix matching

def build_trie(seqs):
    root = {}
    for seq in seqs:
        node = root
        for tok in seq:
            node = node.setdefault(tok, {})
    return root

def longest_prefix_length(root, seq):
    """Return the reusable prefix length of seq in the trie."""
    length = 0
    node = root
    for tok in seq:
        # TODO: Replace the triple-quoted content below with your code
        """If tok is in node, descend and increment the count; otherwise break."""
    return length

trie = build_trie([list("ABCDEF"), list("ABCDXY"), list("ABCZZ")])
assert longest_prefix_length(trie, list("ABCDQ")) == 4
assert longest_prefix_length(trie, list("ABCZ")) == 4
assert longest_prefix_length(trie, list("XYZ")) == 0
print("Exercise 3 passed: a new request pays Prefill only for the unmatched suffix")
